<a href="https://colab.research.google.com/github/scottn66/255-final-project/blob/wp6-colab-gates/notebooks/06_transformer_gates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer gates on Colab: Run all, top to bottom

This notebook runs the checks that must pass before any long run of the band-token transformer (DATA 255 Group 1), in order, then starts the half-resolution screening runs. A failed gate stops the notebook. **Runtime → Change runtime type → GPU, then Runtime → Run all.**

| Step | What it does | Rough time | Stops the notebook when |
|---|---|---|---|
| Setup | clone `BRANCH`, install, mount Drive | 1–2 min | there is no GPU or the clone fails |
| Data | download and unzip SpectralWaste (21 GB zip) to the local disk, or reuse it | 15–40 min, first time in a session | the md5 or the split counts are wrong |
| Gate 0 | smoke: two steps and the parameter count | under 1 min | the parameter line is not 488,487 |
| Gate 1 | memory and step time at full resolution, batch 8, 25 steps | 1–3 min | it runs out of memory at batch 4 too |
| Gate 2 | overfit 4 training cubes that hold all six object classes, 600 epochs | a few minutes | no epoch has all four large classes above 0.9 |
| Gate 3 | screening runs at half resolution, 30 epochs, in the background | hours | (they keep running; the monitor cell reports) |
| Curves | Gate 2 per-class IoU and the screening curves | seconds | |

The times are estimates, not measurements on a Colab GPU. Gate 1 measures them.

**Starting it.** Run all stops once, near the top, for the Google Drive sign-in. Approve it, and the rest runs on its own. Keep the monitor cell running while the screening runs train. If the notebook sits idle, Colab may recycle the machine, and that stops the runs.

**What survives a dead session.** Checkpoints, logs and `gates.json` go to Google Drive (`RUNS`, default `MyDrive/sw_runs`). Open the notebook again and **Run all**:
- Gates 1 and 2 are recorded in `gates.json` under the commit (Gate 1 also under the GPU type), so they are skipped. Set `RERUN_GATES = True` to run them again.
- Screening runs resume from their `last.pt`.
- A run that is still going is not started twice.
- A run whose flags changed under the same tag is refused; give it a new tag.

The cubes stay on the Colab local disk, so a new session downloads them again (or copies `GATES_ZIP` from Drive).

**What counts as a result.** Gates 0 to 2 check the wiring. Gate 2 fits four *training* images and never goes in `docs/RESULTS.md` or on a slide. Gate 3 rows are screening rows: 30 epochs, one seed, the CNN's optimiser settings. They are compared with `cnn_ds2` (0.391 val FG mIoU) and labelled as screening wherever they appear. The runs the hypothesis is judged on come later: after the optimiser-recipe PR (warmup, gradient clipping), and at full resolution.

The decisions (the overfit verdict, the subset of cubes, resumable runs) live in `src/sw/gates.py` and are tested in `tests/test_gates.py`. This notebook and that module were written with AI assistance (Claude Code), then reviewed and run by Scott.

## Settings

Edit here, or set the environment variables (the local test does that). `SCREENING` lists the Gate 3 runs. Each gets its own folder under `RUNS`. Leave a tag unchanged to resume it, and give a new tag to start fresh.

In [ ]:
import os
from pathlib import Path

BRANCH = os.environ.get("SW_BRANCH", "wp6-colab-gates")  # this notebook's branch; it must be on GitHub
REPO_URL = "https://github.com/scottn66/255-final-project.git"
RUNS = Path(os.environ.get("GATES_RUNS", "/content/drive/MyDrive/sw_runs"))  # Drive: checkpoints, logs, gates.json
WORK = Path(os.environ.get("GATES_WORK", "/content/sw_gates"))  # local disk: gates 0 to 2
DATA_DIR = Path(os.environ.get("GATES_DATA", "/content/data"))  # local disk: the cubes (never train from Drive)
GATES_ZIP = os.environ.get("GATES_ZIP", "")  # optional: a copy of spectralwaste_segmentation.zip on Drive
DELETE_ZIP = True  # free 21 GB of local disk once the cubes are unpacked and counted
DEVICE = os.environ.get("GATES_DEVICE", "cuda")
NUM_WORKERS = int(os.environ.get("GATES_WORKERS", "2"))
POLL_SECONDS = int(os.environ.get("GATES_POLL", "300"))  # how often the monitor cell prints

# Gate 1: memory and step time. The full-resolution runs use batch 8 if it fits.
MEM_BATCH = int(os.environ.get("GATES_MEM_BATCH", "8"))
MEM_STEPS = int(os.environ.get("GATES_MEM_STEPS", "25"))
MEM_IMAGES = int(os.environ.get("GATES_MEM_IMAGES", "200"))

# Gate 2: overfit. None = the values in src/sw/gates.py (600, and one resume to 900 if still rising).
OVERFIT_EPOCHS = int(os.environ["GATES_OVERFIT_EPOCHS"]) if os.environ.get("GATES_OVERFIT_EPOCHS") else None
OVERFIT_EXTEND_TO = None

RERUN_GATES = False  # True: run Gates 1 and 2 again even if gates.json already has them for this commit

# Gate 3: screening, at the flags of cnn_ds2 in docs/RESULTS.md.
SCREEN_EPOCHS = int(os.environ.get("GATES_SCREEN_EPOCHS", "30"))
SCREENING = {
    "tf_ds2": ["--model", "transformer", "--downsample", "2", "--batch-size", "4", "--patience", "8"],
}
WAIT_FOR_SCREENING = True  # the monitor cell waits until they finish; interrupting it does not stop the runs

## Setup

On Colab: mount Drive, clone `BRANCH` (or pull it if the clone is already there), install `requirements.txt` (the preinstalled torch stays), and check for a GPU. Locally: use the repo this notebook is in.

In [ ]:
import json
import shutil
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
T_START = time.time()


def sh(cmd):
    """Run a command, show its output, raise if it fails."""
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, check=True)


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = Path("/content/255-final-project")
    if not REPO.is_dir():
        sh(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)])
    else:
        # An earlier clone (for example COLAB.md's, of main) may be on another branch.
        sh(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH])
        try:
            sh(["git", "-C", str(REPO), "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"])
        except subprocess.CalledProcessError as err:
            raise RuntimeError(f"{REPO} has local changes that block switching to {BRANCH}; delete the folder and Run all again") from err
    sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")])
else:
    _env = os.environ.get("REPO_ROOT", "").strip()
    _here = Path.cwd().resolve()
    REPO = Path(_env) if _env else next(p for p in [_here, *_here.parents] if (p / "src" / "sw" / "gates.py").is_file())

os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import torch

from sw import gates

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime → Change runtime type → GPU, then Run all again.")
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
ON_BRANCH = subprocess.run(["git", "branch", "--show-current"], cwd=REPO, capture_output=True, text=True).stdout.strip()
ENV = {**os.environ, "PYTHONPATH": str(REPO / "src")}
RUNS.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
GATES_JSON = RUNS / "gates.json"
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else DEVICE


def recorded(key):
    """This commit's entry for one gate in RUNS/gates.json, or None."""
    data = json.loads(GATES_JSON.read_text()) if GATES_JSON.is_file() else {}
    return data.get(COMMIT, {}).get(key)


def record(key, value):
    """Store one gate's numbers in RUNS/gates.json, under this commit."""
    data = json.loads(GATES_JSON.read_text()) if GATES_JSON.is_file() else {}
    data.setdefault(COMMIT, {})[key] = value
    GATES_JSON.write_text(json.dumps(data, indent=2))


def tail(path, n=25):
    return "\n".join(Path(path).read_text(errors="replace").splitlines()[-n:]) if Path(path).is_file() else "(no log)"


def follow(out, proc=None, every=20):
    """Wait for a run started by gates.launch, printing its latest epoch now and then."""
    alive = (lambda: proc.poll() is None) if proc is not None else (lambda: gates.run_state(out) == "running")
    while alive():
        time.sleep(every)
        try:
            row = gates.read_metrics(out / "metrics.jsonl")[-1]
        except (FileNotFoundError, ValueError):
            continue
        print(
            f"  {out.name}: epoch {row['epoch']}  loss {row['train_loss']:.4f}  "
            f"train FG {row.get('train_fg_miou') or 0:.3f}  val FG {row.get('val_fg_miou') or 0:.3f}",
            flush=True,
        )
    if gates.run_state(out) != "done":
        raise RuntimeError(f"{out.name} stopped before finishing. End of {out / 'train.log'}:\n{tail(out / 'train.log')}")


print(f"repo   {REPO} @ {COMMIT} (branch {ON_BRANCH or 'detached'})")
print(f"torch  {torch.__version__} on {GPU}")
print(f"runs   {RUNS}\nwork   {WORK}\ndata   {DATA_DIR}")
record("session", {"gpu": GPU, "torch": torch.__version__, "started": time.strftime("%Y-%m-%d %H:%M:%S")})

## Data

The cubes go on the Colab local disk: each epoch reads all of them, and Drive is too slow for that. If `SW_ROOT` already holds the full train and val splits, nothing is downloaded. Otherwise `scripts/get_data.sh` downloads the zip (resumable), checks its md5, and unzips it. Setting `GATES_ZIP` to a copy of the zip on Drive copies it instead of downloading.

In [ ]:
from sw.data import SPLIT_COUNTS, SpectralWasteDataset


def counts_ok(root):
    try:
        return all(len(SpectralWasteDataset(root, split=s, random_flip=False)) == SPLIT_COUNTS[s] for s in ("train", "val"))
    except (FileNotFoundError, ValueError, RuntimeError):
        return False


SW_ROOT = Path(os.environ["SW_ROOT"]) if os.environ.get("SW_ROOT") else DATA_DIR / "spectralwaste_segmentation"
ALLOW_PARTIAL = os.environ.get("GATES_ALLOW_PARTIAL") == "1"  # local test only: a small SW_ROOT
if ALLOW_PARTIAL and (SW_ROOT / "hyper").is_dir():
    print(f"SW_ROOT {SW_ROOT}: partial data allowed (local test; not the official split)")
elif counts_ok(SW_ROOT):
    print(f"SW_ROOT {SW_ROOT}: unpacked, train and val counts match")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(DATA_DIR).free / 2**30
    print(f"local disk free: {free:.0f} GiB (zip 21 GB + cubes 24 GB while unzipping)")
    if free < 50:
        print("WARNING: under 50 GiB free; the unzip may run out of space")
    zip_path = DATA_DIR / "spectralwaste_segmentation.zip"
    if GATES_ZIP and Path(GATES_ZIP).is_file() and not zip_path.is_file():
        sh(["cp", GATES_ZIP, str(zip_path)])
    sh(["bash", str(REPO / "scripts" / "get_data.sh"), str(DATA_DIR)])
    SW_ROOT = DATA_DIR / "spectralwaste_segmentation"
    if not counts_ok(SW_ROOT):
        raise RuntimeError(f"{SW_ROOT}: the train/val counts do not match {SPLIT_COUNTS}")
    if DELETE_ZIP:
        zip_path.unlink(missing_ok=True)
os.environ["SW_ROOT"] = ENV["SW_ROOT"] = str(SW_ROOT)

## Gate 0: smoke

Two optimizer steps on a few cubes. The run must not raise, and the parameter line must read `488,487`, the WP6 model at its defaults.

In [ ]:
smoke = subprocess.run(
    [sys.executable, "-m", "sw.train", "--model", "transformer", "--smoke", "--device", DEVICE,
     "--data-root", str(SW_ROOT), "--out-dir", str(WORK / "smoke")],
    capture_output=True, text=True, env=ENV,
)
print(smoke.stdout[-2000:], smoke.stderr[-3000:], sep="\n")
if smoke.returncode != 0:
    raise RuntimeError("Gate 0 failed: the smoke run raised (traceback above)")
if "BandTransformer parameters: 488,487 (0.488M)" not in smoke.stdout:
    raise RuntimeError("Gate 0 failed: the parameter line is not 488,487, so this is not the WP6 model at its defaults")
record("gate0_smoke", "pass")
print("Gate 0 passed")

## Gate 1: memory and step time

The model runs at full resolution, batch 8, for 25 steps, in this process so the CUDA peak is readable. If batch 8 runs out of memory, batch 4 is tried. "Reserved" is what the allocator holds, closer to what `nvidia-smi` shows. The CUDA context adds a few hundred MB on top.

The seconds per step are an upper bound, because `train_seconds` includes the data loader's start-up. The epoch estimate below counts the training steps only, not the validation pass over 167 images. On a T4, a peak above 12 GiB at batch 8 means the attention-memory PR comes before long runs at batch 8.

The numbers are stored in `gates.json` under this commit and GPU type, and the gate is skipped next time. It also waits while a screening run is on the GPU, because a shared GPU would give the wrong numbers.

In [ ]:
import gc
import math

from sw.train import main as train_main


def memcheck(batch):
    out = WORK / f"memcheck_b{batch}"
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    train_main(["--model", "transformer", "--device", DEVICE, "--data-root", str(SW_ROOT),
                "--batch-size", str(batch), "--epochs", "1", "--max-train-steps", str(MEM_STEPS),
                "--max-train-images", str(MEM_IMAGES), "--max-val-images", "8",
                "--num-workers", str(NUM_WORKERS), "--out-dir", str(out)])
    row = gates.read_metrics(out / "metrics.jsonl")[-1]
    info = {"batch": batch, "steps": row["steps"], "train_seconds": row["train_seconds"],
            "seconds_per_step_upper": row["train_seconds"] / row["steps"]}
    if torch.cuda.is_available():
        info["peak_allocated_gib"] = torch.cuda.max_memory_allocated() / 2**30
        info["peak_reserved_gib"] = torch.cuda.max_memory_reserved() / 2**30
    return info


def out_of_memory(err):
    return isinstance(err, torch.cuda.OutOfMemoryError) or "out of memory" in str(err).lower() or "ALLOC_FAILED" in str(err)


MEM = recorded("gate1_memory")
busy = [tag for tag in SCREENING if gates.run_state(RUNS / tag) == "running"]
if MEM is not None and MEM.get("gpu") == GPU and not RERUN_GATES:
    print(f"Gate 1 already measured for {COMMIT} on {GPU}; skipping (RERUN_GATES = True to repeat)")
elif busy:
    print(f"Gate 1 skipped: {busy} are training on this GPU, so the numbers would be wrong. Run it when they finish.")
    MEM = MEM or {}
else:
    MEM = None
    for batch in dict.fromkeys([MEM_BATCH, 4]):
        failed = None
        try:
            MEM = memcheck(batch)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as err:
            if not out_of_memory(err):
                raise
            failed = f"{type(err).__name__}"
        # Free the batch's tensors (the traceback held them) and the allocator cache,
        # so the screening runs below get the memory back.
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if MEM is not None:
            break
        print(f"batch {batch}: out of memory ({failed})")
    if MEM is None:
        raise RuntimeError("Gate 1 failed: out of memory at batch 4 too. The attention-memory PR (9) comes first.")
    MEM["gpu"] = GPU
    # The transformer keeps a short last batch, so an epoch is ceil(514 / batch) steps.
    MEM["steps_per_full_epoch"] = math.ceil(SPLIT_COUNTS["train"] / MEM["batch"])
    MEM["full_epoch_train_seconds_est"] = MEM["steps_per_full_epoch"] * MEM["seconds_per_step_upper"]
    MEM["run_200_epochs_train_hours_est"] = gates.budget_hours(MEM["full_epoch_train_seconds_est"], 200)
    record("gate1_memory", MEM)

if MEM:
    print(f"\nbatch {MEM['batch']}: {MEM['seconds_per_step_upper']:.3f} s/step (upper bound)")
    if "peak_allocated_gib" in MEM:
        print(f"peak allocated {MEM['peak_allocated_gib']:.2f} GiB, reserved {MEM['peak_reserved_gib']:.2f} GiB on {MEM['gpu']}")
        if MEM["batch"] == 8 and MEM["peak_reserved_gib"] > 12:
            print("above 12 GiB at batch 8: on a T4, do the attention-memory PR before long batch-8 runs")
    print(f"one full-resolution epoch: {MEM['steps_per_full_epoch']} steps, about {MEM['full_epoch_train_seconds_est'] / 60:.1f} min of training (plus validation)")
    print(f"one 200-epoch run: about {MEM['run_200_epochs_train_hours_est']:.1f} h of training (plus validation); multiply by the number of runs")

## Gate 2: overfit four training cubes

The transformer must fit four training images when nothing stops it: no augmentation, a constant learning rate, half resolution, and one optimizer step per epoch. The cubes are `gates.OVERFIT_STEMS`, four official training images that together hold all six object classes, linked into a small `SW_ROOT`. (`--max-train-images 4` alone would take the first four training images, and those hold only background, film and bag.)

**Pass:** within 600 epochs, at least one epoch has film, basket, cardboard and bag *all* above 0.9 training IoU. If the run fails while still rising, it resumes once to 900 epochs. Video tape and filament are reported but not judged. They are 1 to 2 pixels wide, and the head without the full-resolution skip blurs them.

**Fail:** a large class never gets there, or the loss is flat. Then the CNN runs on the same cubes and flags, to show where the fault is.

This rule was calibrated on real cubes (see `src/sw/gates.py`). The three transformer seeds passed at epochs 337, 447 and 433; MiniNetSeg (one seed) passed at epoch 43. Training FG mIoU stays under 0.1 for the first 94 to 131 epochs, while the model predicts almost only background. That is expected.

A pass is stored in `gates.json` under this commit, with the log copied to `RUNS/gates/tf_overfit`, and the gate is skipped next time.

In [ ]:
OVERFIT_ROOT = gates.build_subset_root(SW_ROOT, WORK / "overfit_root")
PIXELS = gates.class_pixels(OVERFIT_ROOT, "train")
print("training pixels in the four overfit cubes:", PIXELS)
missing = [name for name in gates.LARGE_CLASSES if PIXELS[name] == 0]
if missing:
    raise RuntimeError(f"Gate 2 cannot run: {missing} are absent from the overfit cubes")


def overfit_args(model, epochs):
    return ["--model", model, "--data-root", str(OVERFIT_ROOT), "--downsample", "2", "--no-aug", "--no-cosine",
            "--max-train-images", "4", "--max-val-images", "4", "--batch-size", "4",
            "--epochs", str(epochs), "--num-workers", "0", "--device", DEVICE]


def judge(out):
    return gates.judge_overfit(gates.read_metrics(out / "metrics.jsonl"), pixels=PIXELS)


def show(name, v):
    print(f"\n{name}: {v['verdict'].upper()}: {v['reason']}")
    first = "n/a" if v["epoch1_fg"] is None else f"{v['epoch1_fg']:.3f}"
    print(f"  epoch 1 train FG mIoU {first} (must be low), epochs run {v['epochs']}")
    for cls, final in v["final"].items():
        best = v["best"][cls]
        role = "judged" if cls in v["classes"] else "reported only"
        print(f"  {cls:11s} best {best if best is None else round(best, 3)!s:>6}   last {final if final is None else round(final, 3)!s:>6}   {role}")


EPOCHS_2 = OVERFIT_EPOCHS or gates.OVERFIT_EPOCHS
EXTEND_2 = OVERFIT_EXTEND_TO or gates.OVERFIT_EXTEND_TO
TF_OVERFIT = WORK / "tf_overfit"  # local disk: 600 checkpoints are not worth a Drive round trip
OVERFIT_LOG = RUNS / "gates" / "tf_overfit"  # the copy that survives the session
previous = recorded("gate2_overfit")
if previous and previous.get("verdict") == "pass" and (OVERFIT_LOG / "metrics.jsonl").is_file() and not RERUN_GATES:
    print(f"Gate 2 already passed for {COMMIT}; reading its log from {OVERFIT_LOG} (RERUN_GATES = True to repeat)")
    VERDICT = judge(OVERFIT_LOG)
else:
    follow(TF_OVERFIT, gates.launch(TF_OVERFIT, overfit_args("transformer", EPOCHS_2), env=ENV))
    VERDICT = judge(TF_OVERFIT)
    if VERDICT["verdict"] == "fail" and VERDICT["rising"] and VERDICT["epochs"] < EXTEND_2:
        print(f"not passed by epoch {VERDICT['epochs']} but still rising: resuming to {EXTEND_2}")
        (TF_OVERFIT / gates.DONE_FILE).unlink()
        follow(TF_OVERFIT, gates.launch(TF_OVERFIT, overfit_args("transformer", EXTEND_2), env=ENV))
        VERDICT = judge(TF_OVERFIT)
    shutil.copytree(TF_OVERFIT, OVERFIT_LOG, ignore=shutil.ignore_patterns("*.pt", "*.tmp"), dirs_exist_ok=True)
    record("gate2_overfit", {k: VERDICT[k] for k in ("verdict", "reason", "first_pass_epoch", "closest_epoch", "epochs", "epoch1_fg", "best", "final")})
show("transformer", VERDICT)

if VERDICT["verdict"] != "pass":
    CNN_OVERFIT = WORK / "cnn_overfit"
    follow(CNN_OVERFIT, gates.launch(CNN_OVERFIT, overfit_args("cnn", EPOCHS_2), env=ENV))
    CNN_VERDICT = judge(CNN_OVERFIT)
    show("CNN control, same cubes and flags", CNN_VERDICT)
    if CNN_VERDICT["verdict"] == "pass":
        hint = "The CNN fits the same cubes, so look at the transformer: token order, tokenizer input scale, the head."
    else:
        hint = "The CNN does not fit them either, so look at the data, the labels, the class weights and the loss."
    raise RuntimeError(f"Gate 2 failed: {VERDICT['reason']}. {hint}")
print("\nGate 2 passed")

## Gate 3: screening runs (background)

Each run in `SCREENING` starts as its own background process (`python -m sw.gates run`), with checkpoints and logs in `RUNS/<tag>/` on Drive. Interrupting a cell does not stop it. Re-running this cell after a disconnect resumes every unfinished run from its `last.pt`, and skips the finished ones (they have `DONE.json`). The data stays on local disk. Several rows share the GPU; at half resolution and batch 4 they fit together.

In [ ]:
PROCS = {}
for tag, flags in SCREENING.items():
    args = [*flags, "--epochs", str(SCREEN_EPOCHS), "--num-workers", str(NUM_WORKERS),
            "--device", DEVICE, "--data-root", str(SW_ROOT)]
    PROCS[tag] = gates.launch(RUNS / tag, args, env=ENV)

### Monitor

Prints every run's progress every `POLL_SECONDS`. Interrupt this cell any time: the runs keep going, and re-running it picks them up. The time left comes from each run's median `epoch_seconds`.

In [ ]:
def status(tag):
    out = RUNS / tag
    state = gates.run_state(out)
    proc = PROCS.get(tag)
    if proc is not None and proc.poll() is None:
        state = "running"
    try:
        rows = gates.read_metrics(out / "metrics.jsonl")
    except (FileNotFoundError, ValueError):
        return state, f"{tag:12s} {state:8s} no epoch finished yet"
    last = rows[-1]
    best = max(rows, key=lambda r: r.get("val_fg_miou") or -1.0)
    sec = gates.median_epoch_seconds(rows) or 0.0
    left = sec * max(0, SCREEN_EPOCHS - int(last["epoch"])) / 3600 if state != "done" else 0.0
    return state, (f"{tag:12s} {state:8s} epoch {last['epoch']:3d}/{SCREEN_EPOCHS}  "
                   f"val FG {last.get('val_fg_miou') or 0:.3f} (best {best.get('val_fg_miou') or 0:.3f} at {best['epoch']})  "
                   f"{sec:.1f} s/epoch  about {left:.1f} h left")


while True:
    report = {tag: status(tag) for tag in SCREENING}
    print(time.strftime("%H:%M:%S"), *[line for _, line in report.values()], sep="\n  ", flush=True)
    states = [state for state, _ in report.values()]
    if all(state == "done" for state in states) or not WAIT_FOR_SCREENING:
        break
    stopped = [tag for tag, (state, _) in report.items() if state in ("new", "partial")]
    if stopped:
        for tag in stopped:
            print(f"\n{tag} is not running and not done. End of its log:\n{tail(RUNS / tag / 'train.log')}")
        raise RuntimeError(f"{stopped} stopped; re-run the Gate 3 cell to resume them from last.pt")
    time.sleep(POLL_SECONDS)

SCREEN = {}
for tag in SCREENING:
    if gates.run_state(RUNS / tag) != "done":
        print(f"{tag}: still training; re-run this cell later for its summary")
        continue
    rows = gates.read_metrics(RUNS / tag / "metrics.jsonl")
    best = max(rows, key=lambda r: r.get("val_fg_miou") or -1.0)
    SCREEN[tag] = {"best_val_fg_miou": best.get("val_fg_miou"), "best_epoch": best["epoch"], "epochs": rows[-1]["epoch"],
                   "per_class_iou_at_best": best.get("per_class_iou"), "median_epoch_seconds": gates.median_epoch_seconds(rows)}
    print(f"{tag}: best val FG mIoU {SCREEN[tag]['best_val_fg_miou'] or 0:.3f} at epoch {best['epoch']} (screening; cnn_ds2 is 0.391)")
if SCREEN:
    record("gate3_screening", {**(recorded("gate3_screening") or {}), **SCREEN})

## Curves

Two separate figures, so the four-cube one never lands on a slide by accident.

1. Gate 2: the per-class training IoU of the four judged classes, with the 0.9 line. Saved to `RUNS/gate2_overfit_wiring_check.png`.
2. Gate 3: validation FG mIoU per epoch against `cnn_ds2` (0.391). Saved to `RUNS/gate3_screening.png`.

For the teaching figure F9, open `05_model_figures.ipynb` with `TRAIN_LOG` set to the line printed below.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

fig, ax = plt.subplots(figsize=(6.5, 3.8))
overfit_rows = gates.read_metrics((OVERFIT_LOG if (OVERFIT_LOG / "metrics.jsonl").is_file() else TF_OVERFIT) / "metrics.jsonl")
epochs = [r["epoch"] for r in overfit_rows]
for cls in gates.LARGE_CLASSES:
    ax.plot(epochs, [r["train_per_class_iou"].get(cls) or 0.0 for r in overfit_rows], label=cls, lw=1.2)
ax.axhline(gates.PASS_IOU, color="0.4", ls=":", lw=1)
if VERDICT["first_pass_epoch"]:
    ax.axvline(VERDICT["first_pass_epoch"], color="0.4", ls="--", lw=1, label=f"all four > 0.9 (epoch {VERDICT['first_pass_epoch']})")
ax.set(title="Gate 2 wiring check: fit 4 training cubes (not a result)", xlabel="epoch (one step each)", ylabel="training IoU", ylim=(0, 1))
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(RUNS / "gate2_overfit_wiring_check.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 3.8))
for tag in SCREENING:
    try:
        rows = gates.read_metrics(RUNS / tag / "metrics.jsonl")
    except (FileNotFoundError, ValueError):
        continue
    ax.plot([r["epoch"] for r in rows], [r.get("val_fg_miou") or 0.0 for r in rows], marker="o", ms=3, label=f"{tag} val")
    if "train_fg_miou" in rows[0]:
        ax.plot([r["epoch"] for r in rows], [r.get("train_fg_miou") or 0.0 for r in rows], ls="--", lw=1, label=f"{tag} train")
ax.axhline(0.391, color="0.4", ls=":", lw=1, label="cnn_ds2 val, 0.391")
ax.set(title=f"Gate 3 screening: half resolution, {SCREEN_EPOCHS} epochs, 1 seed", xlabel="epoch", ylabel="FG mIoU", ylim=(0, 1))
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(RUNS / "gate3_screening.png", dpi=150)
plt.show()

print("TRAIN_LOG=" + ",".join(str(RUNS / tag / "metrics.jsonl") for tag in SCREENING))
print(f"gates.json: {GATES_JSON}")
print(f"notebook ran in {(time.time() - T_START) / 60:.1f} min")

## What to send back, and what comes next

* `RUNS/gates.json`: the GPU, the Gate 1 peak memory and step time, the Gate 2 verdict, and the screening scores, under the commit they ran on.
* `RUNS/<tag>/metrics.jsonl` for each screening run.
* `docs/RESULTS.md` gets one row per screening run, with its flags and the word *screening* in the notes.

Next, before any 200-epoch run: the optimiser-recipe PR (warmup, gradient clipping, cosine floor). On real cubes at a constant learning rate, one overfit seed had a loss spike (0.10 to 1.13 around epoch 550) and recovered. Gate 1's hours per run sets how many seeds fit before Dec 2.